# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) protein abundance dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described by the Croissant schema at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed (uncomment if running locally)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access a few top-level metadata fields
print("\n--- DATASET METADATA OVERVIEW ---")
print(f"Title: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs. Each record set, field, and column has a unique `@id` according to the Croissant schema.

In [ ]:
# List record sets and their fields using their `@id`
print("--- AVAILABLE RECORD SETS AND FIELDS ---\n")
record_sets = list(dataset.metadata.recordSet)
overview = {}
for rs in record_sets:
    print(f"[RecordSet] @id: {rs['@id']} | name: {rs['name']}")
    fields = rs.get('field', [])
    overview[rs['@id']] = []
    for field in fields:
        # Each field is a dict or already dereferenced
        field_obj = field if isinstance(field, dict) else dataset.metadata.get_entity(field)
        print(f"    - [Field] @id: {field_obj['@id']} | name: {field_obj.get('name', field_obj['@id'])}")
        overview[rs['@id']].append(field_obj['@id'])
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use the record set and field `@id`s identified in the overview above.

For this dataset, we'll extract records from the main protein abundance table. If unsure which table to explore, use the first record set listed above.

In [ ]:
# Set up record set and field IDs (edit to analyze a different RecordSet)
# List all available record set @ids
record_set_ids = list(overview.keys())
# We'll choose the first record set by default; change as appropriate
main_record_set_id = record_set_ids[0]

# Load all records for this RecordSet into a pandas DataFrame
print(f"Reading records for RecordSet @id = {main_record_set_id}\n")
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

# Show columns (these are the field @ids)
print("Available columns in DataFrame:")
print(list(df.columns))

# Display the first few rows
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field operations should use the field `@id` as the column key.

Below, we select a numeric field (molecular weight or abundance, using its `@id`) and group by a categorical field (e.g., sample type or accession).

In [ ]:
# Update these to match actual field @ids from the Data Overview above.
# For example, suppose the molecular weight column has @id 'cr:molecular_weight' and accession is 'cr:accession'.

numeric_field_id = None
group_field_id = None

# Guess a numeric field from the columns (use coverage, abundance, or molecular weight if present)
for name in df.columns:
    if any(k in name.lower() for k in ['weight', 'coverage', 'abundance', 'count']):
        numeric_field_id = name
        break
# Guess a group field: accession, sample, or description
for name in df.columns:
    if any(k in name.lower() for k in ['accession', 'sample', 'description']):
        group_field_id = name
        break

if not numeric_field_id:
    raise ValueError("Could not automatically determine a numeric field from the data. Please set 'numeric_field_id' to a valid column name.")
print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Using group field: {group_field_id}")

df_clean = df.copy()

# Convert numeric field to numeric (in case it's string)
df_clean[numeric_field_id] = pd.to_numeric(df_clean[numeric_field_id], errors='coerce')

# Drop NA values for the numeric field
df_clean = df_clean.dropna(subset=[numeric_field_id])

# Remove obvious outliers: keep only values < 99th percentile
upper = df_clean[numeric_field_id].quantile(0.99)
filtered_df = df_clean[df_clean[numeric_field_id] < upper]
print(f"\nFiltered records with {numeric_field_id} < {upper:.2f} (99th percentile): {len(filtered_df)} records\n")

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group field if available
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print("\nGrouped mean by group_field:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between dataset fields. All field references use their `@id` names.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

plt.figure(figsize=(8,4))
plt.title(f"Distribution of {numeric_field_id}")
sns.histplot(filtered_df[numeric_field_id], bins=40, kde=True)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group field is categorical and not too high cardinality, show boxplot
if group_field_id and filtered_df[group_field_id].nunique() < 30:
    plt.figure(figsize=(min(12, 1.5*filtered_df[group_field_id].nunique()),5))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, preview, and analyze the FAIR^2 protein abundance dataset. All data references were made via entity `@id` fields, ensuring robust and standards-based interaction with the Croissant metadata schema.

**Summary:**
- We loaded dataset schema and inspected both record sets and fields using `@id`s.
- We loaded the main record set as a DataFrame, explored its columns, and performed normalization and basic filtering on a numeric field.
- Visualizations highlight the distribution and grouping of key protein metrics.

This Croissant-powered approach enables seamless, machine interpretable exploration and re-use of complex, multi-table datasets.
